# 第11章 検出結果の取得と保存

推論結果から座標、信頼度スコア、クラス情報を取得し、CSVやJSONなどの形式で保存します。

## Resultsオブジェクトの構造

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
results = model('image.jpg')

# 最初の画像の結果を取得
result = results[0]

# 主な属性
print(f"画像パス: {result.path}")              # 元画像のパス
print(f"画像サイズ: {result.orig_shape}")      # (高さ, 幅)
print(f"検出数: {len(result.boxes)}")          # 検出された物体の数


## Boxesオブジェクト

In [ ]:
# 検出されたボックスの情報を取得
boxes = result.boxes

print(f"検出数: {len(boxes)}")
print(f"データ型: {type(boxes)}")


## xyxy形式の取得

In [ ]:
boxes = result.boxes
xyxy = boxes.xyxy

if len(boxes) > 0:
    x1, y1, x2, y2 = xyxy[0]
    print(f"左上: ({x1:.1f}, {y1:.1f})")
    print(f"右下: ({x2:.1f}, {y2:.1f})")
    print(f"幅: {x2 - x1:.1f}, 高さ: {y2 - y1:.1f}")


## xywh形式の取得

In [ ]:
xywh = boxes.xywh

if len(boxes) > 0:
    x_center, y_center, width, height = xywh[0]
    print(f"中心: ({x_center:.1f}, {y_center:.1f})")
    print(f"サイズ: {width:.1f} × {height:.1f}")


## 正規化座標の取得

In [ ]:
# xyxyn形式
xyxyn = boxes.xyxyn
if len(boxes) > 0:
    x1_norm, y1_norm, x2_norm, y2_norm = xyxyn[0]
    print(f"正規化座標(xyxy): ({x1_norm:.3f}, {y1_norm:.3f}) - "
          f"({x2_norm:.3f}, {y2_norm:.3f})")

# xywhn形式
xywhn = boxes.xywhn
if len(boxes) > 0:
    x_norm, y_norm, w_norm, h_norm = xywhn[0]
    print(f"正規化座標(xywh): 中心({x_norm:.3f}, {y_norm:.3f}), "
          f"サイズ({w_norm:.3f}, {h_norm:.3f})")


## 整数座標への変換

In [ ]:
# 整数座標に変換
if len(boxes) > 0:
    x1, y1, x2, y2 = map(int, boxes.xyxy[0])
    print(f"整数座標: ({x1}, {y1}) - ({x2}, {y2})")


## 信頼度スコアの取得

In [ ]:
# 信頼度を取得
confidences = boxes.conf

# 各ボックスの信頼度を表示
for i, conf in enumerate(confidences):
    print(f"Box {i}: 信頼度 {conf:.3f} ({conf*100:.1f}%)")


## クラスIDとクラス名の取得

In [ ]:
# クラスIDを取得
class_ids = boxes.cls

# モデルのクラス名辞書を取得
class_names = model.names

# 各ボックスのクラスを表示
for i, cls_id in enumerate(class_ids):
    class_name = class_names[int(cls_id)]
    print(f"Box {i}: {class_name} (ID: {int(cls_id)})")


## すべての情報をまとめて取得

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
results = model('image.jpg')

boxes = results[0].boxes
class_names = model.names

# すべての検出を処理
for i, box in enumerate(boxes):
    # 座標を取得
    x1, y1, x2, y2 = box.xyxy[0]
    
    # 信頼度を取得
    confidence = box.conf[0]
    
    # クラスIDとクラス名を取得
    class_id = int(box.cls[0])
    class_name = class_names[class_id]
    
    # 幅と高さを計算
    width = x2 - x1
    height = y2 - y1
    
    # 情報を表示
    print(f"[{i}] {class_name}")
    print(f"    信頼度: {confidence:.2f}")
    print(f"    位置: ({x1:.0f}, {y1:.0f}) - ({x2:.0f}, {y2:.0f})")
    print(f"    サイズ: {width:.0f} × {height:.0f}")
    print()


## クラス別の信頼度スコアのしきい値

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
# まず低めの閾値で推論
results = model('image.jpg', conf=0.3)

boxes = results[0].boxes
filtered_boxes = []

for box in boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    class_name = model.names[class_id]
    
    # クラスごとに異なる閾値を適用
    if class_id == 0 and confidence >= 0.3:  # person: 0.3以上
        filtered_boxes.append(box)
        print(f"{class_name}: {confidence:.2f} (採用)")
    elif class_id == 2 and confidence >= 0.5:  # car: 0.5以上
        filtered_boxes.append(box)
        print(f"{class_name}: {confidence:.2f} (採用)")
    elif class_id not in [0, 2] and confidence >= 0.4:  # その他: 0.4以上
        filtered_boxes.append(box)
        print(f"{class_name}: {confidence:.2f} (採用)")
    else:
        print(f"{class_name}: {confidence:.2f} (除外)")

print(f"\n全検出: {len(boxes)}個")
print(f"フィルタ後: {len(filtered_boxes)}個")


## 領域内の検出判定

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
results = model('image.jpg')

boxes = results[0].boxes

# 関心領域を定義（例：画像の中央部分）
image_width = results[0].orig_shape[1]
image_height = results[0].orig_shape[0]

roi_x1 = image_width * 0.3
roi_x2 = image_width * 0.7
roi_y1 = image_height * 0.3
roi_y2 = image_height * 0.7

# 関心領域内の人物をカウント
persons_in_roi = 0
persons_outside_roi = 0

for box in boxes:
    class_id = int(box.cls[0])
    
    if class_id == 0:  # person
        x1, y1, x2, y2 = box.xyxy[0]
        # バウンディングボックスの中心座標を計算
        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2
        
        # 中心が関心領域内にあるか判定
        if roi_x1 <= center_x <= roi_x2 and roi_y1 <= center_y <= roi_y2:
            persons_in_roi += 1
            print(f"中心領域内の人物: ({center_x:.0f}, {center_y:.0f})")
        else:
            persons_outside_roi += 1

print(f"\n中心領域内: {persons_in_roi}人")
print(f"領域外: {persons_outside_roi}人")


## 検出領域の自動切り出し

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

# save_crop=True で検出領域を切り出して保存
results = model('image.jpg', save=True, save_crop=True)

print("検出領域が runs/detect/predict/crops/ に保存されました")


## 複数画像の切り出し

In [ ]:
import glob

image_paths = glob.glob('images/*.jpg')
results = model(image_paths, save=True, save_crop=True)

print(f"{len(image_paths)} 枚の画像を処理しました")


## テキストファイルへのエクスポート

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')

# save_txt=True で検出結果をテキストファイルに保存
results = model('image.jpg', save=True, save_txt=True)


## 信頼度スコア付きでエクスポート

In [ ]:
# 信頼度スコアも含めて保存
results = model('image.jpg', save=True, save_txt=True, save_conf=True)


## CSV形式への変換

In [ ]:
from ultralytics import YOLO
import csv

model = YOLO('yolo26n.pt')
results = model('image.jpg')

boxes = results[0].boxes

with open('detections.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['画像', 'クラス', '信頼度', 'x1', 'y1', 'x2', 'y2'])
    
    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        confidence = float(box.conf[0])
        class_name = model.names[int(box.cls[0])]
        
        writer.writerow([
            'image.jpg',
            class_name,
            f'{confidence:.3f}',
            f'{x1:.1f}',
            f'{y1:.1f}',
            f'{x2:.1f}',
            f'{y2:.1f}'
        ])

print("detections.csv に保存しました")


## JSON形式への変換

In [ ]:
import json

detections = {
    'image': 'image.jpg',
    'detections': []
}

for box in boxes:
    x1, y1, x2, y2 = box.xyxy[0]
    
    detection = {
        'class': model.names[int(box.cls[0])],
        'confidence': round(float(box.conf[0]), 3),
        'bbox': [round(float(x1), 1), round(float(y1), 1), 
                 round(float(x2), 1), round(float(y2), 1)]
    }
    detections['detections'].append(detection)

with open('detections.json', 'w', encoding='utf-8') as f:
    json.dump(detections, f, ensure_ascii=False, indent=2)
